# ЛР3 — трёхмерные графики, вариант 17

Используется `Daily_Water_Intake.csv`. Сеть имеет 4 входа, 3 нейрона в скрытом слое и функцию активации `Tanh`. Полный воспроизводимый запуск находится в `lab3_solution.py`.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
df = pd.read_csv('Daily_Water_Intake.csv').dropna().drop_duplicates().reset_index(drop=True)
df['target'] = df['Hydration Level'].map({'Poor': 0, 'Good': 1})
df['Physical Activity Level'] = df['Physical Activity Level'].map({'Low': 0., 'Moderate': 1., 'High': 2.})
df['Weather'] = df['Weather'].map({'Cold': 0., 'Normal': 1., 'Hot': 2.})
features = ['Age', 'Daily Water Intake (liters)', 'Physical Activity Level', 'Weather']
X_train, X_test, y_train, y_test = train_test_split(df[features], df['target'], test_size=.2, random_state=SEED, stratify=df['target'])
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype('float32')
X_test = scaler.transform(X_test).astype('float32')


In [ ]:
# Корреляции признаков с целевой метрикой
corr = df[features + ['target']].corr()
corr.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1)

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(4, 3), nn.Tanh(), nn.Linear(3, 1))
    def forward(self, x):
        return self.network(x)

model = NeuralNetwork()
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train.to_numpy('float32'))), batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(SEED))
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)
for epoch in range(60):
    model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x_batch).squeeze(1), y_batch)
        loss.backward(); optimizer.step()
torch.save(model.state_dict(), 'water_variant17_Tanh_3.pth')

In [ ]:
# 3D-пространство активаций и разделяющая плоскость
import plotly.graph_objects as go
with torch.no_grad():
    x_all = torch.tensor(scaler.transform(df[features]).astype('float32'))
    before = model.network[0](x_all).numpy()
    after = model.network[1](torch.tensor(before)).numpy()
    prediction = (torch.sigmoid(model.network[2](torch.tensor(after))) >= .5).numpy().ravel().astype(int)
A, B, C = model.network[2].weight.detach().numpy().ravel()
D = -float(model.network[2].bias.detach().numpy()[0])
xx, yy = np.meshgrid(np.linspace(after[:,0].min(), after[:,0].max(), 40), np.linspace(after[:,1].min(), after[:,1].max(), 40))
zz = (D - A*xx - B*yy) / C
fig = go.Figure(go.Scatter3d(x=after[:,0], y=after[:,1], z=after[:,2], mode='markers'))
fig.add_trace(go.Surface(x=xx, y=yy, z=zz, opacity=.45, showscale=False))
fig.write_html('water_variant17_tanh_3_neurons_plane_separator.html', include_plotlyjs='cdn')